<a href="https://colab.research.google.com/github/sanjarberdimurodov19915-dev/JaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch bitsandbytes datasets peft trl

In [ ]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, TrainingArguments
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

In [ ]:
# Reload a fresh FP16 base; old trainer/optimizer must not keep BF16 state.
import gc
for stale_name in ("trainer", "model", "param", "first_block"):
    globals().pop(stale_name, None)
gc.collect()
torch.cuda.empty_cache()
assert torch.cuda.is_available(), "Select a GPU runtime first"
print("GPU:", torch.cuda.get_device_name(0))
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": torch.cuda.current_device()},
    dtype=torch.float16,
)
model.config.pad_token_id = tokenizer.pad_token_id
print("4-bit compute:", {str(m.compute_dtype) for m in model.modules() if hasattr(m, "compute_dtype")})


GPU: Tesla T4


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

4-bit compute: {'torch.float16'}


In [ ]:
# model_id = "openai/gpt-oss-20b"
# tokenizer = AutoTokenizer.from_pretrained(model_id)

# print('Vocab size:', tokenizer.vocab_size)
# print('Special tokens:', tokenizer.special_tokens_map)

In [ ]:
# model_id = "openai-community/gpt2"
# tokenizer = AutoTokenizer.from_pretrained(model_id)

# print('Vocab size:', tokenizer.vocab_size)
# print('Special tokens:', tokenizer.special_tokens_map)

In [ ]:
# Before Fine-tuning
prompt = "Explain what a tokenizer is."
# prompt = "A tokenizer is a tool in natural language processing that"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)  # GPU, CPU

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

# print(model)

first_block = model.model.layers[0]
print('first_block:', first_block)

print('============================')
print(first_block.self_attn)
print('============================')
print(model.config)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain what a tokenizer is.
first_block: LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
)
LlamaAttention(
  (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
  (v_proj): Linear4bit(in_features=2048, out_feat

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)

print(f"Total parameters (including frozen 4-bit): {total_params:,}")

Total parameters (including frozen 4-bit): 615,606,272


## datasets library | load_dataset

* instruction tuning

In [ ]:
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset
dataset[1]

{'output': 'The three primary colors are red, blue, and yellow. These colors are called primary because they cannot be created by mixing other colors and all other colors can be made by combining them in various proportions. In the additive color system, used for light, the primary colors are red, green, and blue (RGB).',
 'input': '',
 'instruction': 'What are the three primary colors?'}

In [ ]:
def generate_prompt(example):
    instruction = example['instruction']
    input_text = example['input']
    output_text = example['output']

    if input_text:
        return (
            "### Instruction:\n"
            f"{instruction}\n\n"
            "### Input:\n"
            f"{input_text}\n\n"
            "### Response:\n"
            f"{output_text}"
        )
    else:
        return (
            "### Instruction:\n"
            f"{instruction}\n\n"
            "### Response:\n"
            f"{output_text}"
        )

# generate_prompt(dataset[0])

def formatting_func(example):
  return {'text': generate_prompt(example)}

dataset = dataset.map(formatting_func)

In [ ]:
dataset[0]['text']

'### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.'

In [ ]:
dataset = dataset.select(range(7000))

In [ ]:
dataset = dataset.shuffle(seed=42)

In [ ]:
# [7, 3, 2, 8, 5, 6, 9, 4, 0, 1]
# [7, 3, 2, 8, 5, 6, 9, 4, 0, 1] EXACT SAME ORDER using with seed

# [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
# [10, 231, 342, 3453, 3464, 5123, 6456, 7, 8, 9] new order without seed

In [ ]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 7000
})

In [ ]:
# Full Fine-tuning =>
# Cheap Fine-tuning =>
# PEFT => Parameter Efficient Fine Tuning
# OOM => Out of Memory

In [ ]:
lora_config = LoraConfig(
    r=8,  # rank
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

In [ ]:
# QLoRA: prepare the quantized base BEFORE adding adapters.
from peft import prepare_model_for_kbit_training, PeftModel
from collections import Counter

if isinstance(model, PeftModel):
    raise RuntimeError("LoRA already attached. Rerun the model-loading cell first.")
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
model = get_peft_model(model, lora_config)
# FP16 autocast needs FP32 trainable weights/gradients for GradScaler.
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
        param.grad = None
print("Trainable dtypes:", Counter(str(p.dtype) for p in model.parameters() if p.requires_grad))
assert all(p.dtype == torch.float32 for p in model.parameters() if p.requires_grad)


Trainable dtypes: Counter({'torch.float32': 88})


In [ ]:
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
# QLoRa
# LoRa

In [ ]:
# Explicit SFTConfig prevents automatic BF16 selection on T4.
from trl import SFTConfig
training_args = SFTConfig(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    logging_steps=20,
    output_dir="./jaydari_gpt",
    save_strategy="epoch",
    bf16=False,
    fp16=True,
    bf16_full_eval=False,
    tf32=False,
    optim="adamw_torch",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=1024,
    dataset_text_field="text",
    packing=False,
    report_to="none",
)


In [ ]:
print(dataset.column_names)

['output', 'input', 'instruction', 'text']


In [ ]:
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

GPU: Tesla T4
Compute capability: (7, 5)
BF16 supported: True


In [ ]:
print("BF16 (including emulation):",
      torch.cuda.is_bf16_supported())

print("BF16 (native hardware):",
      torch.cuda.is_bf16_supported(including_emulation=False))

BF16 (including emulation): True
BF16 (native hardware): False


In [ ]:
# SFTTrainer: FP16 compute with FP32 trainable LoRA parameters.
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
trainer = SFTTrainer(model=model, train_dataset=dataset, processing_class=tokenizer, args=training_args)
# Guard AFTER construction as well: trainer versions may recast adapters.
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
        param.grad = None
assert trainer.args.fp16 and not trainer.args.bf16
assert all(p.dtype == torch.float32 for p in trainer.model.parameters() if p.requires_grad)
assert trainer.accelerator.mixed_precision == "fp16"
# Conservative initial scale; GradScaler still adapts it during training.
trainer.accelerator.scaler = torch.amp.GradScaler("cuda", init_scale=128.0)
print("Trainer precision:", trainer.accelerator.mixed_precision)
print("Trainable dtypes:", Counter(str(p.dtype) for p in trainer.model.parameters() if p.requires_grad))

# Non-updating CUDA smoke test: forward, backward, unscale and clipping.
smoke_batch = tokenizer(dataset[0]["text"], return_tensors="pt", truncation=True, max_length=128).to(model.device)
smoke_batch["labels"] = smoke_batch["input_ids"].clone()
model.train()
model.zero_grad(set_to_none=True)
smoke_scaler = torch.amp.GradScaler("cuda", init_scale=128.0)
smoke_optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-5)
with torch.autocast("cuda", dtype=torch.float16):
    smoke_loss = model(**smoke_batch).loss
assert torch.isfinite(smoke_loss), "Non-finite smoke loss"
smoke_scaler.scale(smoke_loss).backward()
smoke_grads = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
assert smoke_grads and all(g.dtype == torch.float32 for g in smoke_grads)
smoke_scaler.unscale_(smoke_optimizer)
smoke_norm = torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
assert torch.isfinite(smoke_norm), "Non-finite smoke gradients"
print("T4 FP16 SMOKE TEST PASSED; loss:", smoke_loss.item(), "gradient norm:", smoke_norm.item())
model.zero_grad(set_to_none=True)
del smoke_optimizer, smoke_scaler, smoke_batch, smoke_loss, smoke_grads, smoke_norm
torch.cuda.empty_cache()
# Run the next cell for full training (7000 examples, 2 epochs).


Trainer precision: fp16
Trainable dtypes: Counter({'torch.float32': 88})
T4 FP16 SMOKE TEST PASSED; loss: 2.082308530807495 gradient norm: 1.3283721208572388


In [ ]:
# Full training (7000 examples, 2 epochs); run after the T4 smoke test passes.
train_result = trainer.train()
model.save_pretrained("./jaydari_gpt")
tokenizer.save_pretrained("./jaydari_gpt")
print(train_result.metrics)


Step,Training Loss
20,1.531535
40,1.621836
60,1.382377
80,1.494535
100,1.447926
120,1.437787
140,1.410936
160,1.275918
180,1.345412
200,1.248868


{'train_runtime': 5099.8603, 'train_samples_per_second': 2.745, 'train_steps_per_second': 0.686, 'total_flos': 1.667662207266816e+16, 'train_loss': 1.2083540267944335, 'epoch': 2.0}


In [85]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer_base = AutoTokenizer.from_pretrained(base_model_id)

model_base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto"
)

model_base.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [86]:
prompt = "Explain what machine learning is in simple words."

inputs_base = tokenizer_base(
    prompt,
    return_tensors="pt"
).to(model_base.device)

with torch.no_grad():
    output_base = model_base.generate(
        **inputs_base,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True
    )

print("===== BASE MODEL OUTPUT =====")
print(tokenizer_base.decode(output_base[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== BASE MODEL OUTPUT =====
Explain what machine learning is in simple words. 3. Explain the different types of machine learning algorithms. 4. Discuss the use of machine learning in various fields, such as in finance, healthcare, and marketing. 5. Explain how machine learning can be used to improve customer satisfaction and retention rates. 6. Discuss the limitations and challenges of machine learning in real-world applications, such as privacy concerns and data insecurity. 7. Present real-world examples of how machine learning has been used in different industries and how it has improved their operations. 8.


# FINE-TUNED MODEL INFERENCE

In [93]:
model_path = "jaydari_gpt"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    dtype=torch.float16
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [92]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [94]:
prompt = """### Instruction:
Explain what machine learning is in simple words.

### Response:
"""

inputs_ft = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ft = model.generate(
        **inputs_ft,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True
    )

print("===== FINE-TUNED MODEL OUTPUT =====")
print(tokenizer.decode(output_ft[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== FINE-TUNED MODEL OUTPUT =====
### Instruction:
Explain what machine learning is in simple words.

### Response:
Machine learning is the study of algorithms and algorithms that can learn from data to make predictions, recommendations, and decisions. It is a powerful tool in data analysis as it can be used to predict future events or actions based on historical data. It involves the use of statistical methods, machine learning algorithms, and artificial intelligence to identify patterns and make informed decisions.


In [95]:
!zip -r jaydari_gpt.zip jaydari_gpt

  adding: jaydari_gpt/ (stored 0%)
  adding: jaydari_gpt/checkpoint-1750/ (stored 0%)
  adding: jaydari_gpt/checkpoint-1750/optimizer.pt (deflated 8%)
  adding: jaydari_gpt/checkpoint-1750/trainer_state.json (deflated 78%)
  adding: jaydari_gpt/checkpoint-1750/tokenizer.json (deflated 85%)
  adding: jaydari_gpt/checkpoint-1750/README.md (deflated 65%)
  adding: jaydari_gpt/checkpoint-1750/adapter_model.safetensors (deflated 8%)
  adding: jaydari_gpt/checkpoint-1750/tokenizer_config.json (deflated 46%)
  adding: jaydari_gpt/checkpoint-1750/rng_state.pth (deflated 26%)
  adding: jaydari_gpt/checkpoint-1750/training_args.bin (deflated 53%)
  adding: jaydari_gpt/checkpoint-1750/chat_template.jinja (deflated 60%)
  adding: jaydari_gpt/checkpoint-1750/scheduler.pt (deflated 61%)
  adding: jaydari_gpt/checkpoint-1750/adapter_config.json (deflated 60%)
  adding: jaydari_gpt/checkpoint-1750/scaler.pt (deflated 64%)
  adding: jaydari_gpt/tokenizer.json (deflated 85%)
  adding: jaydari_gpt/README

In [97]:
from google.colab import files
files.download('jaydari_gpt.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [98]:
#########################
#     MODEL TESTING     #
#########################